In [1]:
from transformers import pipeline
from pyannote.audio import Pipeline
import re

C:\Users\salem\anaconda3\envs\meeting_ai\lib\site-packages\pyannote\audio\core\io.py:43: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")
C:\Users\salem\anaconda3\envs\meeting_ai\lib\site-packages\pyannote\database\custom.py:62: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
audio_path = "ES2003a.Mix-Headset.wav"

In [3]:
asr = pipeline("automatic-speech-recognition", model="openai/whisper-small", return_timestamps=True)
result = asr(audio_path, generate_kwargs={"language": "en"})

segments = []
for seg in result["chunks"]:
    start, end = seg["timestamp"]
    if start is None or end is None:
        continue
    segments.append({"start": start, "end": end, "text": seg["text"].strip()})
filtered_segments = []

for seg in segments:
    if (seg["end"] - seg["start"]) >= 0.5:
        filtered_segments.append(seg)

segments = filtered_segments

print("Transcription Done")

C:\Users\salem\anaconda3\envs\meeting_ai\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
C:\Users\salem\anaconda3\envs\meeting_ai\lib\site-packages\transformers\generation\configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `None` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Transcription Done


In [ ]:
import os

diar_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    use_auth_token=os.environ["HF_TOKEN"]  # set via .env / environment — never hardcode tokens
)

diarization = diar_pipeline(
    audio_path,
    min_speakers=2,
    max_speakers=8
)

speaker_segments = []
for turn, _, speaker in diarization.itertracks(yield_label=True):
    speaker_segments.append({
        "start": turn.start,
        "end": turn.end,
        "speaker": speaker
    })

print("Diarization Done")

In [11]:
def get_speaker(start, end, speaker_segments):
    max_overlap = 0
    best_speaker = "Unknown"

    for seg in speaker_segments:
        s_start = seg["start"]
        s_end = seg["end"]

        # calculate overlap
        overlap = max(0, min(end, s_end) - max(start, s_start))

        if overlap > max_overlap:
            max_overlap = overlap
            best_speaker = seg["speaker"]

    return best_speaker

In [12]:
def extract_names(text):
    import re
    text = text.strip()

    patterns = [
        r"my name is ([A-Z][a-z]+ [A-Z][a-z]+)",
        r"my name's ([A-Z][a-z]+ [A-Z][a-z]+)",
        r"this is ([A-Z][a-z]+ [A-Z][a-z]+)",
        r"we have ([A-Z][a-z]+ [A-Z][a-z]+)",
        r"lastly we have ([A-Z][a-z]+ [A-Z][a-z]+)"
    ]

    for p in patterns:
        match = re.search(p, text)
        if match:
            return match.group(1)

    return None

In [13]:
speaker_name_candidates = {}

for seg in segments:
    speaker = get_speaker(seg["start"], seg["end"], speaker_segments)
    name = extract_names(seg["text"])

    if name:
        if speaker not in speaker_name_candidates:
            speaker_name_candidates[speaker] = []

        speaker_name_candidates[speaker].append(name)
        
from collections import Counter

speaker_name_map = {}

for speaker, names in speaker_name_candidates.items():
    most_common = Counter(names).most_common(1)[0][0]
    speaker_name_map[speaker] = most_common

for spk in set([s["speaker"] for s in speaker_segments]):
    if spk not in speaker_name_map:
        speaker_name_map[spk] = spk 

In [14]:
final_output = []
for seg in segments:
    speaker = get_speaker(seg["start"], seg["end"], speaker_segments)
    speaker = speaker_name_map.get(speaker, speaker)
    line = f"[{seg['start']:.2f}-{seg['end']:.2f}] {speaker}: {seg['text']}"
    final_output.append(line)
    print(line)

[0.00-5.00] Adam Dugard: Okay, well I think we're ready to begin.
[5.00-10.00] Adam Dugard: Right, my name's Adam Dugard, we're here because of Real Reaction.
[10.00-13.00] Adam Dugard: We have in the group...
[13.00-18.00] Dave Cochran: Ebenezer at Demisoy, would you like me to spoke with you?
[18.00-20.00] Dave Cochran: Yeah, go for it mate.
[20.00-22.00] Dave Cochran: NEZ.
[22.00-23.00] Dave Cochran: NEZ.
[23.00-24.00] Dave Cochran: ER.
[24.00-26.00] Dave Cochran: Ebenezer, and your role is?
[26.00-28.00] Dave Cochran: I'm the marketing expert.
[0.00-1.70] Unknown: but you're a marketing expert.
[1.70-2.20] Adam Dugard: OK.
[5.36-6.36] Adam Dugard: Next we have.
[6.36-8.56] Adam Dugard: Tarek Ramar, T-A-R-I-K.
[8.56-10.32] Adam Dugard: T-R-I-K.
[10.32-11.88] Adam Dugard: And your role in this is?
[11.88-13.08] Adam Dugard: Industrial designer.
[13.08-14.04] Dave Cochran: Industrial designer.
[19.72-25.04] Dave Cochran: And lastly we have Dave Cochran.
[25.04-27.76] Dave Cochran: And

In [15]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

full_text = " ".join([seg["text"] for seg in segments])

def chunk_text(text, max_words=200):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), max_words):
        chunk = " ".join(words[i:i+max_words])
        chunks.append(chunk)
    
    return chunks


chunks = chunk_text(full_text)
partial_summaries = []

for chunk in chunks:
    summary = summarizer(
        chunk,
        max_length=120,
        min_length=40,
        do_sample=False
    )[0]["summary_text"]
    
    partial_summaries.append(summary)

final_summary = " ".join(partial_summaries)

print("\n FINAL SUMMARY:\n")
print(final_summary)

C:\Users\salem\anaconda3\envs\meeting_ai\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Your max_length is set to 120, but your input_length is only 10. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)



 FINAL SUMMARY:

Adam Dugard, from Real Reaction, leads the panel of experts. The group will be working on a new remote control design. The aim is to make it original, trendy and user-friendly. The 3K phase is going to be covering the user requirements spec, technical functions, working design. Second seems to be conceptual components, properties, materials. And the last one is a detailed analysis of our design so far. You should also have your little pal mic on as well. We can clip them on. OK. Just somewhere around. the other it's just across here that's it is this supposed to be clear? I think so yeah I'll follow you. Monkeys have attitude which I think is a good thing and from the point of view of the study of human evolution they end up probably it's a terribly interesting. Also not quite as feared as your average tiger, but cats are one of my favourite animals. We're hoping to sell them for 25. if we're aiming for 50 million euros we're going to have to be selling an awful lot o